# Exporting

## Settings

In [4]:
PROJECT = "/content/drive/MyDrive/projects/asr"
V1_BEST = f"{PROJECT}/checkpoints/nemotron-3.5-asr-streaming-0.6b/best.nemo"
CHUNK_MS = 560

WORK_DIR = "/content/export"
OUT_DIR = f"{PROJECT}/exports/nemotron-v1-{CHUNK_MS}ms"
SCRIPT_URL = (
    "https://raw.githubusercontent.com/k2-fsa/sherpa-onnx/master/"
    "scripts/nemo/nemotron-3.5-asr-streaming-0.6b/export_onnx.py"
)

In [2]:
!apt-get install -y -qq libsndfile1 ffmpeg
!pip install -q Cython packaging
!pip install -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@main" onnx onnxruntime

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import json
import os
import shutil
from pathlib import Path

import onnx
from google.colab import drive

drive.mount("/content/drive")
assert Path(V1_BEST).exists(), f"not found: {V1_BEST}"

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
print("working in", os.getcwd())

Mounted at /content/drive
working in /content/export


## Download and change export script

In [4]:
!wget -q -O export_onnx.py {SCRIPT_URL}

script = Path("export_onnx.py").read_text()
old = "asr_model = nemo_asr.models.ASRModel.from_pretrained(model_name=model_name)"
new = f'asr_model = nemo_asr.models.ASRModel.restore_from({V1_BEST!r}, map_location="cpu")'
assert script.count(old) == 1, "the script changed upstream: check its model-loading line"
Path("export_onnx.py").write_text(script.replace(old, new))

print([line.strip() for line in Path("export_onnx.py").read_text().splitlines() if "restore_from" in line])

['asr_model = nemo_asr.models.ASRModel.restore_from(\'/content/drive/MyDrive/projects/asr/checkpoints/nemotron-3.5-asr-streaming-0.6b/best.nemo\', map_location="cpu")']


## Export

In [5]:
!python export_onnx.py --chunk-size-ms {CHUNK_MS} 2>&1 | tee export.log

[NeMo I 2026-11-23 19:47:49 mixins:197] Tokenizer SentencePieceTokenizer initialized with 13087 tokens
[NeMo W 2026-11-23 19:47:55 modelPT:309] You tried to register an artifact under config key=tokenizer.model_path but an artifact for it has already been registered.
[NeMo W 2026-11-23 19:47:55 modelPT:309] You tried to register an artifact under config key=tokenizer.vocab_path but an artifact for it has already been registered.
[NeMo I 2026-11-23 19:47:55 mixins:197] Tokenizer SentencePieceTokenizer initialized with 13087 tokens
[NeMo W 2026-11-23 19:48:03 modelPT:192] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /content/manifests/train.json
    sample_rate: 16000
    use_lhotse: true
    batch_duration: 100
    use_bucketing: true
    num_buckets: 30
    quadratic_duration: 15
    bucket_buffer_size: 10000
    shuffle_

### Checking export

In [8]:
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
for name in ("encoder.onnx", "encoder.data", "decoder.onnx", "joiner.onnx", "tokens.txt", "export.log", "export_onnx.py"):
    shutil.copy(Path(WORK_DIR) / name, Path(OUT_DIR) / name)
    print("copied", name)

copied encoder.onnx
copied encoder.data
copied decoder.onnx
copied joiner.onnx
copied tokens.txt
copied export.log
copied export_onnx.py


## Copy to drive

### INT8

In [1]:
!pip install -q onnx onnxruntime

In [2]:
import json
import shutil
from pathlib import Path

import onnx
from google.colab import drive
from onnxruntime.quantization import QuantType, quantize_dynamic

drive.mount("/content/drive")
!free -h

Mounted at /content/drive
               total        used        free      shared  buff/cache   available
Mem:            52Gi       1.8Gi        48Gi       2.8Mi       3.7Gi        51Gi
Swap:             0B          0B          0B


### Quantization

In [5]:
out = Path(WORK_DIR) / str(CHUNK_MS)
out.mkdir(parents=True, exist_ok=True)
for name in ("encoder.onnx", "encoder.data", "decoder.onnx", "joiner.onnx", "tokens.txt"):
    shutil.copy(Path(OUT_DIR) / name, out / name)

for m in ("encoder", "decoder", "joiner"):
    quantize_dynamic(
        model_input=str(out / f"{m}.onnx"),
        model_output=str(out / f"{m}.int8.onnx"),
        weight_type=QuantType.QUInt8,
    )
    print(m, "done")

  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 9
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 3
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 3
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 3
    }
    dim {
      dim_value: 2
    }
  }
}
.
  elem_type: 7
  shape {
    dim {
      dim_value: 4


encoder done


decoder done
joiner done


In [6]:
log = (Path(OUT_DIR) / "export.log").read_text()
assert f"restored from {V1_BEST}" in log, "the export did not load v1"
assert f"att_context_size [56, {CHUNK_MS // 80 - 1}]" in log, "wrong attention context"

expected = ["encoder.onnx", "encoder.data", "decoder.onnx", "joiner.onnx",
            "encoder.int8.onnx", "decoder.int8.onnx", "joiner.int8.onnx", "tokens.txt"]
missing = [name for name in expected if not (out / name).exists()]
assert not missing, f"missing files: {missing}"

meta = {p.key: p.value for p in onnx.load(str(out / "encoder.onnx"), load_external_data=False).metadata_props}
prompts = json.loads(meta["prompt_dictionary"])
assert meta["chunk_size_ms"] == str(CHUNK_MS), meta["chunk_size_ms"]
assert prompts["hi-IN"] == 6, prompts.get("hi-IN")

n_tokens = sum(1 for _ in open(out / "tokens.txt", encoding="utf-8"))
assert n_tokens == int(meta["vocab_size"]) + 1, (n_tokens, meta["vocab_size"])

for name in expected:
    print(f"{name:20s} {(out / name).stat().st_size / 2**20:8.1f} MB")
int8_mb = sum((out / f"{m}.int8.onnx").stat().st_size for m in ("encoder", "decoder", "joiner")) / 2**20
print(f"INT8 total: {int8_mb:.0f} MB (target: 700 MB or less)")

encoder.onnx             40.3 MB
encoder.data           2340.7 MB
decoder.onnx             57.0 MB
joiner.onnx              36.1 MB
encoder.int8.onnx       627.1 MB
decoder.int8.onnx        14.3 MB
joiner.int8.onnx          9.1 MB
tokens.txt                0.1 MB
INT8 total: 650 MB (target: 700 MB or less)


In [7]:
for m in ("encoder", "decoder", "joiner"):
    shutil.copy(out / f"{m}.int8.onnx", Path(OUT_DIR) / f"{m}.int8.onnx")
print(sorted(p.name for p in Path(OUT_DIR).iterdir()))

['decoder.int8.onnx', 'decoder.onnx', 'encoder.data', 'encoder.int8.onnx', 'encoder.onnx', 'export.log', 'export_onnx.py', 'joiner.int8.onnx', 'joiner.onnx', 'tokens.txt']


## Install sherpa-onnx

In [8]:
!pip install -q sherpa-onnx soundfile jiwer
import sherpa_onnx
print("sherpa-onnx", sherpa_onnx.__version__)

sherpa-onnx 2.0.3


Clips for the check

In [9]:
import random
import sys
import zipfile

sys.path.insert(0, f"{PROJECT}/src")
from hinglish_asr.metrics import score
from hinglish_asr.utils import Normalizer

PROMPT = "hi-IN"
N_CLIPS = 50
V1_HYPS = f"{PROJECT}/results/nemotron-3.5-asr-streaming-0.6b-finetuned/hyps-560ms.jsonl"
CHECK_DIR = Path("/content/check_clips")

norm = Normalizer(f"{PROJECT}/data/lists/loanwords.csv", f"{PROJECT}/data/lists/transliterations.csv")
with open(V1_HYPS, encoding="utf-8") as f:
    v1 = [json.loads(line) for line in f]
candidates = [r for r in v1 if r["hyp"].strip()]
check = random.Random(0).sample(candidates, N_CLIPS)

wanted = {r["utt_id"] + ".wav" for r in check}
CHECK_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile(f"{PROJECT}/openslr104_test.zip") as z:
    for member in z.infolist():
        if Path(member.filename).name in wanted:
            (CHECK_DIR / Path(member.filename).name).write_bytes(z.read(member))
missing = [r["utt_id"] for r in check if not (CHECK_DIR / f"{r['utt_id']}.wav").exists()]
assert not missing, f"{len(missing)} clips not found in the zip, e.g. {missing[:3]}"
print(f"{len(check)} clips ready")

50 clips ready


Decode with sherpa-onnx

In [10]:
import time

import numpy as np
import soundfile as sf


def make_recognizer(precision):
    suffix = ".int8" if precision == "int8" else ""
    return sherpa_onnx.OnlineRecognizer.from_transducer(
        tokens=str(out / "tokens.txt"),
        encoder=str(out / f"encoder{suffix}.onnx"),
        decoder=str(out / f"decoder{suffix}.onnx"),
        joiner=str(out / f"joiner{suffix}.onnx"),
        num_threads=4,
        decoding_method="greedy_search",
    )


def decode(recognizer, wav_path):
    samples, sample_rate = sf.read(wav_path, dtype="float32")
    stream = recognizer.create_stream()
    stream.set_option("language", PROMPT)
    stream.accept_waveform(sample_rate, samples)
    stream.accept_waveform(sample_rate, np.zeros(int(0.66 * sample_rate), dtype=np.float32))
    stream.input_finished()
    while recognizer.is_ready(stream):
        recognizer.decode_stream(stream)
    return recognizer.get_result(stream).strip()


audio_s = sum(sf.info(str(CHECK_DIR / f"{r['utt_id']}.wav")).duration for r in check)
onnx_hyps = {}
for precision in ("fp32", "int8"):
    recognizer = make_recognizer(precision)
    start = time.time()
    onnx_hyps[precision] = [decode(recognizer, str(CHECK_DIR / f"{r['utt_id']}.wav")) for r in check]
    elapsed = time.time() - start
    print(f"{precision}: {elapsed:.0f} s for {audio_s:.0f} s of audio, CPU RTF {elapsed / audio_s:.3f}")
    del recognizer

fp32: 100 s for 360 s of audio, CPU RTF 0.279
int8: 88 s for 360 s of audio, CPU RTF 0.244


## Compare with v1 in NeMo

In [11]:
refs = [r["ref"] for r in check]
nemo = [r["hyp"] for r in check]

print(f"NeMo v1 against references: {score(refs, nemo, norm)}")
for precision, hyps in onnx_hyps.items():
    same = sum(norm.for_scoring(a) == norm.for_scoring(b) for a, b in zip(nemo, hyps))
    print(f"\n{precision}:")
    print(f"  identical to NeMo after normalization: {same} of {len(check)}")
    print(f"  difference from NeMo (NeMo output as reference): {score(nemo, hyps, norm)}")
    print(f"  against references: {score(refs, hyps, norm)}")
    print(f"  empty outputs: {sum(not h for h in hyps)}")

shown = 0
for r, a, b in zip(check, nemo, onnx_hyps["fp32"]):
    if norm.for_scoring(a) != norm.for_scoring(b) and shown < 5:
        print(f"\n{r['utt_id']}\n  NeMo: {a}\n  FP32: {b}")
        shown += 1

NeMo v1 against references: {'wer': 0.44174135723431496, 't_wer': 0.44174135723431496}

fp32:
  identical to NeMo after normalization: 20 of 50
  difference from NeMo (NeMo output as reference): {'wer': 0.22996515679442509, 't_wer': 0.22822299651567945}
  against references: {'wer': 0.49167733674775926, 't_wer': 0.49039692701664533}
  empty outputs: 3

int8:
  identical to NeMo after normalization: 18 of 50
  difference from NeMo (NeMo output as reference): {'wer': 0.2229965156794425, 't_wer': 0.21777003484320556}
  against references: {'wer': 0.4788732394366197, 't_wer': 0.47503201024327785}
  empty outputs: 1

957491_4xyIm2P6Xzlin341_0007
  NeMo: यह एक बड़ा platform software है जिसका अर्थ है कि यह विभिन्न operating system पर दिखाई देता है
  FP32: यह एक बड़ा platform software है जिसका अर्थ है कि यह विभिन्न operating system पर

318923_NhV290ew2hDJfVqc_0116
  NeMo: निम्न link पर दबाएं
  FP32: निम्न link पर

918821_jFTbEEy3BCO5axj5_0002
  NeMo: linux में installation यानि संस्थापन होता ह

## checks

In [13]:
import pandas as pd


def decode(recognizer, wav_path, tail_s=0.66):
    samples, sample_rate = sf.read(wav_path, dtype="float32")
    stream = recognizer.create_stream()
    stream.set_option("language", PROMPT)
    stream.accept_waveform(sample_rate, samples)
    stream.accept_waveform(sample_rate, np.zeros(int(tail_s * sample_rate), dtype=np.float32))
    stream.input_finished()
    while recognizer.is_ready(stream):
        recognizer.decode_stream(stream)
    return recognizer.get_result(stream).strip()


rows_out = []
for precision in ("fp32", "int8"):
    recognizer = make_recognizer(precision)
    for tail_s in (0.66, 1.5, 3.0):
        hyps = [decode(recognizer, str(CHECK_DIR / f"{r['utt_id']}.wav"), tail_s) for r in check]
        rows_out.append({
            "precision": precision,
            "tail_s": tail_s,
            "identical": sum(norm.for_scoring(a) == norm.for_scoring(b) for a, b in zip(nemo, hyps)),
            "diff_from_nemo": round(score(nemo, hyps, norm)["wer"], 3),
            "wer_vs_refs": round(score(refs, hyps, norm)["wer"], 3),
            "empty": sum(not h for h in hyps),
        })
    del recognizer

print(f"NeMo v1 WER against references on these clips: {score(refs, nemo, norm)['wer']:.3f}")
pd.DataFrame(rows_out)

NeMo v1 WER against references on these clips: 0.442


,precision,tail_s,identical,diff_from_nemo,wer_vs_refs,empty
0,fp32,0.66,20,0.230,0.492,3
1,fp32,1.50,20,0.228,0.493,3
2,fp32,3.00,20,0.228,0.493,3
3,int8,0.66,18,0.223,0.479,1
4,int8,1.50,18,0.220,0.481,1
5,int8,3.00,18,0.220,0.481,1
